this notebook is written after the learning rate experiments, and with the slx proof of concept notebook in mind but before it was finished. 

In [61]:
import numpy as np
import polars as pl
from project_paths import paths, project_root
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, GroupKFold
from imd_features.spatial_utils import fetch_spatial_support_data
from imd_features.cross_validate import cross_validate

In [62]:
engineered_features_path = project_root / "data" / "output" / "engineered_rates_v2_c3ad0464.parquet"

df_features = pl.read_parquet(engineered_features_path)
target_df = pl.read_parquet(paths.reference)

df = df_features.join(target_df, on="lsoa_code", how="inner")



feature_cols = [c for c in df.columns if c not in ("lsoa_code", "score", "rank")]
lsoa_codes = df.get_column("lsoa_code").to_list()
X = df.select(feature_cols).to_numpy()
y = df.select("score").to_numpy().ravel()

In [63]:
print(X.shape)
print(y.shape)
# feature_cols

(268, 54)
(268,)


In [64]:
feature_cols

['crime_rate_per_1000',
 'violent_crime_rate',
 'asb_rate',
 'burglary_rate',
 'drugs_rate',
 'resolution_rate',
 'uc_claim_rate',
 'uc_nwr_rate',
 '%_claims_planfw',
 '%_claims_sfw',
 'Overall',
 'Overall (walking)',
 'Overall (public transport)',
 'Healthcare (walking)',
 'nearest_shop',
 'nearest_hospital',
 'nearest_pharmacy',
 'nearest_school',
 'nearest_bank',
 'nearest_fast_food',
 'nearest_pub',
 'nearest_gambling',
 'lsoa_mean_price',
 'lsoa_median_price',
 'lsoa_price_inequality',
 'lsoa_max_price',
 'transactions_per_capita',
 'flats_proportion',
 'terraced_proportion',
 'detached_proportion',
 'new_build_proportion',
 'freehold_proportion',
 'count_healthcare_access_1000',
 'count_education_skills_1000',
 'count_essential_services_1000',
 'count_transport_public_1000',
 'count_financial_services_1000',
 'count_retail_commerce_1000',
 'count_fast_food_takeaway_1000',
 'count_food_dining_1000',
 'ratio_fast_food_takeaway_to_food_dining_1000',
 'ratio_alcohol_gambling_to_finan

In [65]:
W, groups_10, aligned_codes = fetch_spatial_support_data(
    lsoa_codes=lsoa_codes,
    boundaries_path=paths.polygons,
    lookup_path=paths.lads,
    n_clusters_per_city=10,
)

In [66]:
print(aligned_codes == lsoa_codes)

True


In [67]:
model = Ridge(alpha=1.0)
standard_cv = KFold(n_splits=5, shuffle=True, random_state=123)

baseline = cross_validate(X, y, model, standard_cv)

In [68]:
print(f"R2: {baseline['r2_mean']}")
print(f"R2 std: {baseline['r2_std']}")
print(f"rsme: {baseline['rmse_mean']}")
print(f"rsme std: {baseline['rmse_std']}")
print(f"spearman: {baseline['spearman_mean']}")
print(f"spearman std: {baseline['spearman_std']}")

R2: 0.9520923448875575
R2 std: 0.013174320194286058
rsme: 3.3932454053916308
rsme std: 0.3706437465988592
spearman: 0.9649126973655274
spearman std: 0.006146652657496174


In [69]:
k_values = [5, 8, 10, 15]
spatial_results = {k: {"result": None, "gap": None} for k in k_values}

for k in k_values:
    _, groups_k, _ = fetch_spatial_support_data(
        lsoa_codes=lsoa_codes,
        boundaries_path=paths.polygons,
        lookup_path=paths.lads,
        n_clusters_per_city=k,
    )

    spatial_cv = GroupKFold(n_splits=5)
    result = cross_validate(X, y, model, spatial_cv, groups=groups_k)
    spatial_results[k]["result"] = result
    spatial_results[k]["gap"] = baseline["r2_mean"] - result["r2_mean"]


In [70]:
from pprint import pprint
pprint(spatial_results)

{5: {'gap': 0.18036644916283506,
     'result': {'r2_mean': 0.7717258957247224,
                'r2_per_fold': [0.26863824706237627,
                                0.9824572705318299,
                                0.7970613134495411,
                                0.9300794547520445,
                                0.88039319282782],
                'r2_std': 0.25885563894661645,
                'rmse_mean': 5.594509791289289,
                'rmse_per_fold': [np.float64(10.809388536930006),
                                  np.float64(2.632452733406253),
                                  np.float64(5.146273982740106),
                                  np.float64(3.4313572252550446),
                                  np.float64(5.953076478115033)],
                'rmse_std': 2.862776860979475,
                'spearman_mean': 0.9035683539841157,
                'spearman_per_fold': [np.float64(0.7817689048364316),
                                      np.float64(0.9825740978570169

In [75]:
import pandas as pd

rows = [{"cv": "standard", "k": "-", **{f"{m}": baseline[f"{m}_mean"] for m in ["r2", "rmse", "spearman"]}}]


for k, r in spatial_results.items():
    result = r.get("result")

    rows.append({
        "cv": "spatial",
        "k": k,
        **{f"{m}": result[f"{m}_mean"] for m in ["r2", "rmse", "spearman"]},
    })

summary = pd.DataFrame(rows)
summary["r2_gap"] = rows[0]["r2"] - summary["r2"]
summary.index.name = None
summary
# print(summary.to_latex())

,cv,k,r2,rmse,spearman,r2_gap
0,standard,-,0.952092,3.393245,0.964913,0.000000
1,spatial,5,0.771726,5.594510,0.903568,0.180366
2,spatial,8,0.924303,3.530992,0.934765,0.027789
3,spatial,10,0.926099,3.673788,0.927302,0.025993
4,spatial,15,0.931149,3.512913,0.944185,0.020943


In [76]:


per_fold = pd.DataFrame({
    f"spatial_k={k}": result.get("result").get("r2_per_fold")
    for k, result in spatial_results.items()
})
per_fold["standard"] = baseline["r2_per_fold"]
per_fold.index.name = None
per_fold
# print(per_fold.to_latex())

,spatial_k=5,spatial_k=8,spatial_k=10,spatial_k=15,standard
0,0.268638,0.926256,0.904557,0.945093,0.928732
1,0.982457,0.875077,0.948506,0.875664,0.965733
2,0.797061,0.918966,0.893988,0.933763,0.963029
3,0.930079,0.921512,0.950226,0.976489,0.954285
4,0.880393,0.979707,0.933221,0.924735,0.948682


# Spatial CV

this notebook is set up after the results of the learning curve experiments exloring overfitting. The premise was that the expansion of the dataset had improved the model as much as possible (the learning rate had plataued) but there was still some overfitting present, and I theorised that this must be coming from spatial autocorrelation (leakage) or non independance of the observations. 

Spatial cross validation has been implemented so that it can be used to evaluate ridge regression, and so that we can compare the evalutations of the ridge model under standard cv and spatial cv. If there is a gap in the metrics, we can learn something about the scale of spatial leakage in the model. 

This notebook shows:

- spatial leakage is real: it is always positive at every value of k
- it is very small. It would seem that the overall assessment of the model being good enough holds.


The different values of k are supposed to act as sensitivity testing. Using a different value of k should mean that the clusters produced are slightly different, so we can see how much changing the cluster affects the outcome. I think these all look similar and there is no real pattern. i would have thought that the gap reduces as k increases, as at the extreme (k=n) you have just one cluster and its equivilent to standard cross validation. The gap actually increases from 5 to 8. I wonder if this is just instability, and running again with a higher number of folds would produce a different pattern.

we can also see that the folds are less stable for spatial cv than for standard (higher std of r2 and rmse). This makes sense but it is worth knowing that maybe 7 folds should be used to report final performance of the model with spatial cv.

practically, these results suggest that spatial leakage is not driving overfitting.


At this point, its probably fair to put the gap in the test and train r2 down to multicollinearity rather than a structural problem. 


## update

the above was written using the full input parquet as the feature frame - that is, the engineered rates but also the population values. The results were:

```
| cv       | k  | r2    | rmse  | spearman | r2_gap |
|----------|----|-------|-------|----------|--------|
| standard | -  | 0.987 | 1.683 | 0.990    | 0.000  |
| spatial  | 5  | 0.957 | 2.649 | 0.971    | 0.029  |
| spatial  | 8  | 0.953 | 2.938 | 0.957    | 0.034  |
| spatial  | 10 | 0.979 | 1.948 | 0.978    | 0.008  |
| spatial  | 15 | 0.972 | 2.131 | 0.971    | 0.015  |
```

afterwards, for a fair comparison, i switched to using the engineered_rates feature config that was produced in a previous notebook. This has changed the results to what is now visible in the notebook. 

# spatial CV (draft 2)

this notebook is set up after the results of the learning curve experiments exloring overfitting. The premise was that the expansion of the dataset had improved the model as much as possible (the learning rate had plataued) but there was still some overfitting present, and I theorised that this must be coming from spatial autocorrelation (leakage) or non independance of the observations. 

Spatial cross validation has been implemented so that it can be used to evaluate ridge regression, and so that we can compare the evalutations of the ridge model under standard cv and spatial cv. If there is a gap in the metrics, we can learn something about the scale of spatial leakage in the model. 

This notebook shows:

- spatial leakage is real: it is always positive at every value of k
- at small values of k, the model performs much more poorly, and we can see that it completely failed in one fold (r2 = 0.2). 
- the gap between non spatial cv and spatial cv closes as k gets larger. As k increases, clusters of lsoas are smaller and there are more of them. This increases the chance that an lsoa being evaluated (tested) is nearby an lsoa that was in the training data. Whereas when k is small, there are only 5 clusters per city and its possible that in one fold most of the lsoas evaluated (tested) had none of its neighbours of even neighbour's neighbours in the training data.

This tells us that the model definitly is reliant on spatial proximity to get good results. When we holdout small fragments, there are enough nearby training examples that the model can predict well in the gaps. When we hold out large regions, the model can fail. 

the sensitivity to k is very interesting. some quick envelope maths: id guess that a cluster of ~12to15 lsoas (at k=15) is about 1.5 km across (in my head 3 * 500m if you get three lsoas lined up end to end in the cluster) and a cluster of ~30to40 lsoas (at k=5) is about 3 to 5 km across ( a multiple of before). This gives us an idea of the scale of an area you can hold out and the model can still fill the gaps well. It could be interesting to explore this properly with actual cluster sizes and some visualizations (prob second paper territory tbh).

another future investigation would be the fold that completely failed. If it was when test lsoas were in one particular city, which city was it? This could be a novel way of comparing structural similarities/disimilarities of cities/areas. 

Now that we have established that there is some spatial dependency, we will have to think about how this affects how we test with a combined 2019 and 2025 dataset - hold out the same lsoas in both timeframes, only hold out in one timeframe, etc.? we will then have to handle combined spatial and temporal leakage effects. Its possible that the spatial dependency is actually linked to some structure that changes over time. If the model is strongly fitted to something like this (as a random example, road network patterns - then a load of roads become one way or switch direction between the time points) then it will perform poorly in the other timeframe since that structure is not explicitly in the input. 



the practical implication is - yes the model is overfit, but it is overfit specifically on bristol. Since our premise is "can we nowcast for bristol", this is not an issue. Our "out of sample" data points will all be in bristol when we implement the model to produce our presentation results. If the question was "how well can the model predict deprivation in a city the model has never seen before?", then the answer is that it performs poorly. This is a relevant finding when we discuss future research, and possibly even important to discuss whether our extra data came from appropriate cities, but it doesn't speak directly to our project scope and research question. 


This is actually an expected result imo, and it helps us quantify how much the model depends on having nearby training examples. I think the more interesting question now is - why did this pattern show up on the feature config (engineered rates has about 55 features) and not on the full parquet? 

what this tells us is that the full parquet of features generalizes better across cities cities than the reduced/selected config

# lessons for next steps

use spatial cv with k=8,10,15 for standard evaluation in all experiments going forward. This is lower than standard cv and more honest

The k=5 shows us when the model fails, and we need to be clear about not claiming that the model can predict brand new regions (fine)

we know have a clean baseline for comparing ridge to slx


in a technical write up we would include something like "We adopt spatial CV with k=10 as the default evaluation strategy for all subsequent modelling, as it provides a conservative but realistic performance estimate (R2 = 0.926 vs 0.952 under standard CV) for our within city nowcasting use case."

